# Glass Rose

**Author:** Iman Mossavat · Fontys ICT

## What creates a glass rose?

A rose window is built from radial symmetry. A single half-sector of seeds is designed, then copied around the center by rotation and mirroring. After that, a color recipe and a few lighting stages turn flat cells into glowing glass.

![Final glass rose artwork](../assets/glass_rose_preview.png)


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from math_vis import seeds_rose, make_cmap, glass, finish

SEED = 9
SIZE = 420
N_FOLD = 10
RINGS = 4
rng = np.random.default_rng(SEED)
palette = make_cmap(['#1a0b2e', '#4a1c40', '#b83b5e', '#f08a5d', '#f9ed69'])

## Symmetric seed layout

The `seeds_rose` function places points in one half-sector and then mirrors and rotates them. This guarantees that the whole design has the same number of folds as a rose window.

In [ ]:
# 1. Symmetric seed layout and ids
seeds, ids = seeds_rose(N_FOLD, RINGS, SIZE, SIZE, rng)

fig, ax = plt.subplots(figsize=(7, 7))
scatter = ax.scatter(seeds[:,0], seeds[:,1], c=ids, cmap='tab20', s=15)
ax.set_xlim(0,SIZE); ax.set_ylim(0,SIZE); ax.set_aspect('equal'); ax.axis('off')
ax.set_title(f'Rose seeds: {len(seeds)} points, ids color one half-sector')
plt.tight_layout(); plt.show()

> **Question:** Why is it enough to design *half* of one sector? What symmetry operations complete the full circle?

<details><summary>Answer</summary>

DRAFT: author to review

`seeds_rose` builds one angular sector and then copies it `N_FOLD` times around the centre, so the whole window has `N_FOLD`-fold rotational symmetry. Within each sector the seed pattern is also mirrored across the sector bisector, so designing half a sector is enough: the other half is obtained by reflection. This is why the parameters only describe the geometry in `[0, π/N_FOLD]`.

</details>

## From geometry to color

Color depends on both distance from the center and angle around it. The same symmetric seed layout can look completely different by changing the color recipe.

In [ ]:
# 2. Rose color recipe
r = np.hypot(seeds[:,0] - SIZE/2, seeds[:,1] - SIZE/2) / (0.46 * SIZE)
th = np.arctan2(seeds[:,1] - SIZE/2, seeds[:,0] - SIZE/2)
jitter = rng.normal(0, 0.05, ids.max()+1)[ids]
t_cell = np.clip(
    0.55 * r
    + 0.22 * np.cos(N_FOLD * th)
    + 0.10 * np.cos(2 * N_FOLD * th)
    + jitter
    + 0.1,
    0, 1,
)

flat = (palette(t_cell)[:,:3] * 255).astype(np.uint8)
# assign to pixels for preview
from scipy.spatial import cKDTree
X, Y = np.meshgrid(np.arange(SIZE)+0.5, np.arange(SIZE)+0.5)
cell = cKDTree(seeds).query(np.column_stack([X.ravel(), Y.ravel()]))[1]
flat_img = flat[cell].reshape(SIZE, SIZE, 3)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(flat_img, origin='lower'); ax.axis('off')
ax.set_title('Flat rose color recipe')
plt.tight_layout(); plt.show()

> **Question:** If you increased the weight on the `np.cos(N_FOLD * th)` term, would the color bands become narrower or wider? Predict, then try it.

<details><summary>Answer</summary>

DRAFT: author to review

The `np.cos(N_FOLD * th)` term has `N_FOLD` cycles around the centre. Giving it more weight strengthens the radial, petal-like colour bands and makes adjacent sectors contrast more. It would look more like a ringed or flower-petal window, while lowering it leaves smoother, more continuous colour gradients.

</details>

## Lighting stages

The `glass` function adds edge darkening, internal glow, texture, bloom, and a circular crop. Each stage moves the image from flat cells toward stained glass.

In [ ]:
# 3. Lighting stages
img, stages = glass(seeds, t_cell, palette, SIZE, SIZE, rng=rng,
                    facet=0.45, glow=0.4, lead_px=4.5, lead_var=0.35,
                    bevel=0.35, texture=0.04, bloom=0.2, stages=True)

# apply circular crop
yy, xx = np.mgrid[0:SIZE, 0:SIZE]
mask = np.clip((0.47 * SIZE - np.hypot(xx - SIZE/2, yy - SIZE/2)) / 3, 0, 1)[..., None]
for key in stages:
    stages[key] = (stages[key] * mask + np.array([14,11,18]) * (1 - mask)).astype(np.uint8)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for (name, img), ax in zip(stages.items(), axes.ravel()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis('off')
# hide any unused panels in the 2x3 grid
for ax in axes.ravel()[len(stages):]:
    ax.axis('off')
plt.tight_layout(); plt.show()

> **Question:** Which lighting stage do you think contributes most to the "glass" feeling — the glow, the bevel, or the circular crop?

<details><summary>Answer</summary>

DRAFT: author to review

In the lighting pipeline, `facet` adds a directional tilt per cell and `glow` brightens the cell interior relative to its edges (`0.45 - clip(d1/R)`). The `glow` term is what makes the flat cells look rounded and glassy, because it simulates light passing through the thicker centre. The bloom pass adds the soft halo that reinforces the "glowing glass" feel.

</details>

## Experiments: change one thing at a time

Try changing `N_FOLD`, `RINGS`, or the palette. `N_FOLD` controls how many times the half-sector repeats; `RINGS` controls how many rows of seeds appear in the original half-sector. Change one value, then re-run the cells from the parameter cell onward.

In [ ]:
# 4. Final render
final = finish(stages['final'], name='glass_rose.png', grain=2.5, vignette=0.5, display=False)

# Embed a lightweight preview
from IPython.display import display
from PIL import Image
preview = Image.fromarray(final).resize((520, 520))
display(preview)

## What did we learn?

- **Radial symmetry** can be created by mirroring and rotating a small prototype.
- **Color** can be driven by geometric coordinates like radius and angle.
- **Lighting stages** combine to create the illusion of material.
- Small parameter changes (`N_FOLD`, palette) can produce very different moods.

**Open questions:**
1. What would happen if the mirror symmetry were removed and only rotation were used?
2. How would the image change if the color recipe also depended on cell area?

**Further experiment:** Try a monochrome palette with strong bloom, or a high `N_FOLD` value with only one ring.

## Further reading

- Conceptual background: [Symmetry through repetition](conceptualGuide.md#5-symmetry-through-repetition) in `conceptualGuide.md`.
- Implementation details: [Glass rose (rose window)](recipes.md#7-glass-rose-rose-window) and [Glass lighting model](recipes.md#8-glass-lighting-model-glass) in `recipes.md`.
- Same cells, seen as a graph: `demo_proximity_graphs.ipynb` (Voronoi walls are the dual of the Delaunay links, which contain the Gabriel, relative-neighbour and minimum-spanning graphs).
- Same glass, with leaf veins as the lead: `demo_vein_glass.ipynb`.